# Regression & Multilinear Models in Quantitative Finance

A hands-on guide to regression techniques used in modern quantitative finance — from foundational OLS to regularized models, factor models, and rolling regression. Every section includes mock data, visualizations, and practical examples tied to real-world quant applications.

## 1. Import Libraries & Generate Mock Financial Data

Load all dependencies and generate a realistic multi-factor dataset of stock returns, macro variables, and fundamental factors.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.diagnostic import het_breuschpagan, het_white
from statsmodels.stats.stattools import durbin_watson
from sklearn.linear_model import Ridge, Lasso, LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 6)

# ── Mock Dataset: 500 days of multi-factor stock data ──────────────────────────
n = 500
dates = pd.date_range(start='2022-01-01', periods=n, freq='B')

# True factor exposures (used to generate returns)
TRUE_BETA_MKT   = 1.20   # market beta
TRUE_BETA_SMB   = 0.40   # small-cap tilt
TRUE_BETA_HML   = 0.25   # value tilt
TRUE_BETA_MOM   = -0.15  # slight negative momentum
TRUE_ALPHA      = 0.0002  # daily alpha

# Factor returns (market + Fama-French-style)
market_returns   = np.random.normal(0.0004, 0.012, n)
smb_factor       = np.random.normal(0.0001, 0.006, n)
hml_factor       = np.random.normal(0.0001, 0.005, n)
mom_factor       = np.random.normal(0.0002, 0.007, n)
risk_free        = np.full(n, 0.00008)   # ~2% annual, daily

# Stock excess returns (true DGP + noise)
epsilon = np.random.normal(0, 0.008, n)
stock_returns = (TRUE_ALPHA
                 + TRUE_BETA_MKT   * (market_returns - risk_free)
                 + TRUE_BETA_SMB   * smb_factor
                 + TRUE_BETA_HML   * hml_factor
                 + TRUE_BETA_MOM   * mom_factor
                 + epsilon)

# Macro variables (for macro-regression examples)
gdp_growth   = np.random.normal(0.005, 0.003, n).cumsum()
inflation    = 0.02/252 + np.random.normal(0, 0.001, n).cumsum()
credit_sprd  = 0.02 + np.random.normal(0, 0.002, n).cumsum()

# Fundamental / firm-level variables (cross-sectional)
n_stocks = 100
pe_ratio       = np.random.lognormal(np.log(20), 0.5, n_stocks)
pb_ratio       = np.random.lognormal(np.log(3), 0.4, n_stocks)
roe            = np.random.normal(0.12, 0.06, n_stocks)
debt_equity    = np.random.lognormal(np.log(0.8), 0.6, n_stocks)
market_cap_log = np.random.normal(8, 1.5, n_stocks)  # log market cap

# Forward returns (dependent variable for cross-sectional regression)
fw_returns = (-0.002 * np.log(pe_ratio)     # value: lower P/E → higher return
              + 0.015 * roe                  # profitability
              - 0.005 * debt_equity          # leverage penalised
              + 0.003 * market_cap_log       # slight size effect
              + np.random.normal(0, 0.04, n_stocks))

df_ts = pd.DataFrame({
    'stock_ret': stock_returns,
    'mkt_excess': market_returns - risk_free,
    'smb': smb_factor,
    'hml': hml_factor,
    'mom': mom_factor,
    'gdp': gdp_growth,
    'inflation': inflation,
    'credit_sprd': credit_sprd,
}, index=dates)

df_cross = pd.DataFrame({
    'fw_return': fw_returns,
    'log_pe': np.log(pe_ratio),
    'log_pb': np.log(pb_ratio),
    'roe': roe,
    'debt_equity': debt_equity,
    'log_mktcap': market_cap_log,
})

print("✓ Time-series dataset shape:", df_ts.shape)
print("✓ Cross-sectional dataset shape:", df_cross.shape)
print("\nTime-series dataset (first 5 rows):")
print(df_ts.head())


ImportError: cannot import name '_lazywhere' from 'scipy._lib._util' (/Users/yevgeniy/Development/Ext/anaconda3/envs/qf/lib/python3.11/site-packages/scipy/_lib/_util.py)

## 2. Simple Linear Regression (OLS) — CAPM Beta Estimation

### What Is It?
OLS (Ordinary Least Squares) finds the line $\hat{y} = \alpha + \beta x$ that minimises the sum of squared residuals:
$$\hat{\beta} = \frac{\text{Cov}(R_i, R_m)}{\text{Var}(R_m)}, \quad \hat{\alpha} = \bar{R}_i - \hat{\beta}\bar{R}_m$$

### Why Quants Need It
- **CAPM Beta Estimation**: The single most-used regression in finance. Every equity risk model starts with $R_i - R_f = \alpha + \beta(R_m - R_f) + \varepsilon$.
- **Attribution**: A portfolio manager proves that outperformance is genuine alpha, not just hidden market beta hiding inside a "market-neutral" fund.
- **Hedge Ratio Calculation**: A derivatives desk regresses an option's P&L on the underlying to find the delta-hedge ratio via OLS, especially for illiquid instruments without closed-form Greeks.
- **ETF Replication**: ETF providers regress the basket price on the index to compute tracking error and verify the replication quality daily.

**Interpreting CAPM Regression:**
| Term | Meaning |
|------|---------|
| α (intercept) | Return unexplained by market — genuine skill (or luck) |
| β (slope) | Sensitivity to market; β > 1 amplifies market moves |
| R² | Fraction of stock return variance explained by the market |
| p-value of α | Test whether alpha is statistically significant |

In [2]:
# ── Simple OLS: CAPM Beta Estimation ─────────────────────────────────────────
y = df_ts['stock_ret'].values
X_capm = sm.add_constant(df_ts['mkt_excess'].values)

ols_capm = sm.OLS(y, X_capm).fit()
alpha_hat = ols_capm.params[0]
beta_hat  = ols_capm.params[1]
r2        = ols_capm.rsquared

print("=" * 65)
print("CAPM OLS REGRESSION RESULTS")
print("=" * 65)
print(ols_capm.summary())

print(f"\nTrue  α = {TRUE_ALPHA:.6f}  |  Estimated α = {alpha_hat:.6f}")
print(f"True  β = {TRUE_BETA_MKT:.4f}      |  Estimated β = {beta_hat:.4f}")

# ── Visualisation ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter + regression line
ax = axes[0]
mkt_ex = df_ts['mkt_excess'].values
ax.scatter(mkt_ex, y, alpha=0.3, s=15, color='steelblue', label='Daily returns')
x_line = np.linspace(mkt_ex.min(), mkt_ex.max(), 200)
ax.plot(x_line, alpha_hat + beta_hat * x_line, color='red', linewidth=2,
        label=f'OLS: α={alpha_hat:.5f}, β={beta_hat:.3f}')
ax.set_xlabel('Market Excess Return', fontweight='bold')
ax.set_ylabel('Stock Excess Return', fontweight='bold')
ax.set_title(f'CAPM Regression  (R²={r2:.3f})', fontweight='bold')
ax.legend()
ax.grid(alpha=0.3)

# Residuals over time
ax = axes[1]
ax.plot(df_ts.index, ols_capm.resid, linewidth=1, color='purple', alpha=0.7)
ax.axhline(0, color='red', linestyle='--')
ax.set_title('OLS Residuals Over Time', fontweight='bold')
ax.set_ylabel('Residual', fontweight='bold')
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n✓ Annualised alpha: {alpha_hat * 252 * 100:.2f}%")
print(f"✓ Beta (market sensitivity): {beta_hat:.4f}")
print(f"✓ R²: {r2:.4f} — market explains {r2*100:.1f}% of return variance")


NameError: name 'df_ts' is not defined

## 3. Multiple Linear Regression — Fama-French 4-Factor Model

### What Is It?
Multiple linear regression extends OLS to several predictors simultaneously:
$$R_i - R_f = \alpha + \beta_{\text{MKT}}(R_m - R_f) + \beta_{\text{SMB}}\cdot\text{SMB} + \beta_{\text{HML}}\cdot\text{HML} + \beta_{\text{MOM}}\cdot\text{MOM} + \varepsilon$$

The OLS estimator in matrix form: $\hat{\boldsymbol{\beta}} = (X^TX)^{-1}X^T y$

### Why Quants Need It
- **Risk Decomposition**: A single-factor CAPM beta blends together market risk, size risk, value risk, and momentum — all distinct economic sources of return. MLR separates them, giving a cleaner picture of true alpha.
- **Factor Investing (Smart Beta ETFs)**: iShares, Vanguard, and DFA use multi-factor regression to build portfolios that explicitly tilt toward factors (value, momentum, quality) with known historical risk premia.
- **Performance Attribution (Brinson-Hood-Beebower style)**: A pension fund attribution report decomposes total return into contributions from each factor exposure. Without MLR, you cannot tell whether outperformance is from value, momentum, or genuine stock selection.
- **Risk Model Calibration (Barra/Axioma)**: Commercial risk models (Barra USE4, Axioma Worldwide) are fundamentally multi-factor regressions across hundreds of factors. Portfolio construction at every major investment firm uses these daily.

### What Each Coefficient Tells You
| Factor | Positive β means... | Application |
|--------|-------------------|-------------|
| MKT (market) | Stock amplifies market moves | Hedging, leverage control |
| SMB (small-minus-big) | Stock behaves like small caps | Size risk assessment |
| HML (high-minus-low B/P) | Stock behaves like value stocks | Style drift monitoring |
| MOM (momentum) | Stock rides winners | Momentum strategy construction |

In [ ]:
# ── Fama-French 4-Factor OLS ─────────────────────────────────────────────────
factor_cols = ['mkt_excess', 'smb', 'hml', 'mom']
X_ff4 = sm.add_constant(df_ts[factor_cols].values)
ols_ff4 = sm.OLS(y, X_ff4).fit()

print("=" * 65)
print("FAMA-FRENCH 4-FACTOR MODEL RESULTS")
print("=" * 65)
print(ols_ff4.summary())

# Compare true vs estimated betas
true_betas  = [TRUE_ALPHA, TRUE_BETA_MKT, TRUE_BETA_SMB, TRUE_BETA_HML, TRUE_BETA_MOM]
est_betas   = ols_ff4.params
labels      = ['α', 'β_MKT', 'β_SMB', 'β_HML', 'β_MOM']

print("\n" + "=" * 50)
print(f"{'Factor':<10} {'True':>10} {'Estimated':>12} {'t-stat':>10} {'p-value':>10}")
print("-" * 50)
for lab, tr, est, tstat, pv in zip(labels, true_betas, est_betas,
                                    ols_ff4.tvalues, ols_ff4.pvalues):
    sig = '***' if pv < 0.001 else '**' if pv < 0.01 else '*' if pv < 0.05 else ''
    print(f"{lab:<10} {tr:>10.5f} {est:>12.5f} {tstat:>10.3f} {pv:>10.4f} {sig}")

print(f"\nCAPM R² = {ols_capm.rsquared:.4f}  vs  4-Factor R² = {ols_ff4.rsquared:.4f}")
print(f"Improvement from adding SMB/HML/MOM: +{(ols_ff4.rsquared - ols_capm.rsquared)*100:.2f}%")

# ── Coefficient plot with confidence intervals ────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
coef_df = pd.DataFrame({
    'coef': ols_ff4.params[1:],
    'lower': ols_ff4.conf_int()[0][1:],
    'upper': ols_ff4.conf_int()[1][1:],
    'true':  true_betas[1:]
}, index=factor_cols)

y_pos = np.arange(len(coef_df))
ax.barh(y_pos, coef_df['coef'], xerr=[coef_df['coef']-coef_df['lower'],
                                        coef_df['upper']-coef_df['coef']],
        color='steelblue', alpha=0.7, capsize=5, label='Estimated β (95% CI)')
ax.scatter(coef_df['true'], y_pos, color='red', zorder=5, s=80, label='True β')
ax.set_yticks(y_pos)
ax.set_yticklabels(factor_cols, fontsize=11)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Factor Loadings with 95% Confidence Intervals', fontweight='bold')
ax.set_xlabel('Coefficient', fontweight='bold')
ax.legend()
ax.grid(alpha=0.3, axis='x')

# Actual vs Fitted
ax = axes[1]
fitted = ols_ff4.fittedvalues
ax.scatter(fitted, y, alpha=0.3, s=15, color='steelblue')
lim = max(abs(fitted).max(), abs(y).max())
ax.plot([-lim, lim], [-lim, lim], 'r--', linewidth=2, label='Perfect fit')
ax.set_xlabel('Fitted Values', fontweight='bold')
ax.set_ylabel('Actual Returns', fontweight='bold')
ax.set_title('Actual vs Fitted Returns', fontweight='bold')
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()


## 4. OLS Assumptions & Gauss-Markov Theorem

### What Is It?
OLS is **BLUE** (Best Linear Unbiased Estimator) *only if* the Gauss-Markov assumptions hold:

| Assumption | Requirement | Violated By |
|-----------|------------|-------------|
| GM1 | Linearity in parameters | Omitted nonlinear relationships |
| GM2 | Random sampling / exogeneity | Endogenous regressors |
| GM3 | No perfect multicollinearity | Highly correlated features |
| GM4 | Zero conditional mean: $E(\varepsilon|X)=0$ | Omitted variables, model misspecification |
| GM5 | Homoskedasticity: $\text{Var}(\varepsilon|X)=\sigma^2$ | Volatility clustering, leverage effects |
| GM6 | No autocorrelation: $\text{Cov}(\varepsilon_i,\varepsilon_j)=0$ | Time series data |

### Why Quants Need to Check These
- **Biased Betas**: If GM4 fails (e.g., you omit the HML factor but the stock has a value tilt), the CAPM beta is biased. You attribute to alpha what is really compensation for value risk.
- **Wrong Standard Errors → Wrong Significance Tests**: Heteroskedasticity (GM5) and autocorrelation (GM6) produce artificially small standard errors. A factor appears statistically significant when it isn't — leading to false alpha discovery. The 2013 Harvey/Liu paper showed >50% of published "factors" were false discoveries partly for this reason.
- **Unstable Predictions**: Near-multicollinearity (GM3) makes coefficient estimates numerically unstable — changing 1 data point can flip a coefficient sign. This is dangerous when building factor models or running quantitative screens.

In [ ]:
# ── Residual Diagnostic Plots (4-factor model) ───────────────────────────────
resid = ols_ff4.resid
fitted = ols_ff4.fittedvalues

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Residuals vs Fitted
ax = axes[0, 0]
ax.scatter(fitted, resid, alpha=0.4, s=15, color='steelblue')
ax.axhline(0, color='red', linestyle='--', linewidth=1.5)
ax.set_xlabel('Fitted Values', fontweight='bold')
ax.set_ylabel('Residuals', fontweight='bold')
ax.set_title('Residuals vs Fitted\n(check: random cloud around 0 = good)', fontweight='bold')
ax.grid(alpha=0.3)

# 2. Q-Q plot of residuals
ax = axes[0, 1]
stats.probplot(resid, dist='norm', plot=ax)
ax.set_title('Q-Q Plot of Residuals\n(check: points on diagonal = normality holds)', fontweight='bold')
ax.grid(alpha=0.3)

# 3. Scale-Location (sqrt|residuals| vs fitted)
ax = axes[1, 0]
ax.scatter(fitted, np.sqrt(np.abs(resid)), alpha=0.4, s=15, color='green')
z = np.polyfit(fitted, np.sqrt(np.abs(resid)), 1)
x_line = np.linspace(fitted.min(), fitted.max(), 100)
ax.plot(x_line, np.poly1d(z)(x_line), color='red', linewidth=2)
ax.set_xlabel('Fitted Values', fontweight='bold')
ax.set_ylabel('√|Residuals|', fontweight='bold')
ax.set_title('Scale-Location Plot\n(check: flat red line = homoskedasticity)', fontweight='bold')
ax.grid(alpha=0.3)

# 4. Residuals over time (autocorrelation check)
ax = axes[1, 1]
ax.plot(df_ts.index, resid, linewidth=1, color='purple', alpha=0.7)
ax.axhline(0, color='red', linestyle='--')
ax.fill_between(df_ts.index, resid, 0, alpha=0.2, color='purple')
ax.set_title('Residuals Over Time\n(check: no patterns = no autocorrelation)', fontweight='bold')
ax.set_ylabel('Residual', fontweight='bold')
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Formal tests
dw = durbin_watson(resid)
_, bp_pvalue, _, _ = het_breuschpagan(resid, X_ff4)
sw_stat, sw_pvalue = stats.shapiro(resid[:50])

print("=" * 55)
print("REGRESSION ASSUMPTION TESTS")
print("=" * 55)
print(f"Durbin-Watson (autocorrelation) : {dw:.4f}  {'✓ OK (near 2)' if 1.5 < dw < 2.5 else '⚠ Potential autocorrelation'}")
print(f"Breusch-Pagan p-value (hetero)  : {bp_pvalue:.4f}  {'✓ Homoskedastic' if bp_pvalue > 0.05 else '⚠ Heteroskedasticity detected'}")
print(f"Shapiro-Wilk p-value (normality): {sw_pvalue:.4f}  {'✓ Normal residuals' if sw_pvalue > 0.05 else '⚠ Non-normal residuals'}")


## 5. Multicollinearity — Variance Inflation Factor (VIF)

### What Is It?
Multicollinearity occurs when predictors are highly correlated. The **Variance Inflation Factor** quantifies how much a predictor's variance is inflated by correlation with others:
$$\text{VIF}_j = \frac{1}{1 - R^2_j}$$
where $R^2_j$ is obtained by regressing $X_j$ on all other predictors.

**Rule of thumb**: VIF > 5 is concerning; VIF > 10 is severe.

### Why Quants Need It
- **Unstable Factor Betas**: In a risk model with correlated factors (e.g., P/E ratio, P/B ratio, and EV/EBITDA all included), VIF > 10 means tiny perturbations in the sample produce enormous swings in estimated loadings. Barra and Axioma orthogonalise factors precisely to solve this.
- **Misleading Attribution**: A multi-factor attribution report may show a huge positive beta on one value factor and a large negative beta on a correlated value sub-factor. Both are artefacts of multicollinearity. The net effect is correct, but individual factor explanations are wrong.
- **Signal Redundancy in Quant Screens**: A quantitative equity model that includes 50 highly correlated accounting ratios (all variations of leverage) cannot reliably rank stocks. Detecting VIF > 10 drives factor consolidation or dimensionality reduction (PCA, covered later).

In [ ]:
# ── VIF on 4-factor model ─────────────────────────────────────────────────────
X_ff4_df = sm.add_constant(df_ts[factor_cols])

vif_data = pd.DataFrame({
    'Feature': factor_cols,
    'VIF': [variance_inflation_factor(X_ff4_df.values, i+1) for i in range(len(factor_cols))]
})
print("=" * 40)
print("VARIANCE INFLATION FACTORS")
print("=" * 40)
print(vif_data.to_string(index=False))

# ── Now demonstrate with correlated factors ───────────────────────────────────
print("\n── Demonstrating high-VIF scenario ──")
# Add a nearly-redundant factor (smb ≈ 0.9*smb + noise)
df_ts['smb_corr'] = 0.90 * df_ts['smb'] + np.random.normal(0, 0.001, n)

factor_cols_bad = ['mkt_excess', 'smb', 'hml', 'mom', 'smb_corr']
X_bad = sm.add_constant(df_ts[factor_cols_bad])
vif_bad = pd.DataFrame({
    'Feature': factor_cols_bad,
    'VIF': [variance_inflation_factor(X_bad.values, i+1) for i in range(len(factor_cols_bad))]
})
print(vif_bad.to_string(index=False))
print("\n⚠ High VIF on smb / smb_corr — these factors are nearly collinear")

# ── Visualise ─────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Correlation heatmap (good model)
ax = axes[0]
corr = df_ts[factor_cols].corr()
sns.heatmap(corr, annot=True, fmt='.3f', cmap='coolwarm', center=0,
            square=True, ax=ax, cbar_kws={'label': 'Pearson r'})
ax.set_title('Factor Correlation Matrix\n(Good Model — Low VIF)', fontweight='bold')

# VIF bar chart
ax = axes[1]
colors = ['green' if v < 5 else 'orange' if v < 10 else 'red' for v in vif_bad['VIF']]
ax.barh(vif_bad['Feature'], vif_bad['VIF'], color=colors, alpha=0.8)
ax.axvline(5, color='orange', linestyle='--', linewidth=2, label='VIF=5 (warning)')
ax.axvline(10, color='red', linestyle='--', linewidth=2, label='VIF=10 (severe)')
ax.set_xlabel('VIF', fontweight='bold')
ax.set_title('VIF — High Collinearity Example\n(smb ≈ smb_corr)', fontweight='bold')
ax.legend()
ax.grid(alpha=0.3, axis='x')

plt.tight_layout()
plt.show()


## 6. Heteroskedasticity — Breusch-Pagan & White Tests

### What Is It?
**Homoskedasticity** means residuals have constant variance: $\text{Var}(\varepsilon_i | X_i) = \sigma^2$.  
**Heteroskedasticity** means residual variance changes with the level of predictors or time — extremely common in financial returns data.

$$\text{Breusch-Pagan: } H_0 = \text{Homoskedastic}, \quad \text{reject if } p < 0.05$$

### Why Quants Need It
- **Standard Error Inflation / Deflation**: Heteroskedastic errors cause OLS standard errors to be wrong. In equity research, a beta estimate may appear highly significant (t-stat = 3.2) but the true t-stat under correct HC standard errors is only 1.4 — below significance.
- **VaR Model Validation**: Regulatory backtesting of VaR models (Basel traffic light test) checks whether the frequency of VaR breaches is uniform over time. Clustering of breaches (more in volatile periods) signals heteroskedasticity in the risk model.
- **Robust Standard Errors**: The fix for heteroskedasticity is Huber-White "sandwich" robust standard errors (HC3 in statsmodels). CFA and FRM exams test this. All regression outputs in professional quant research should use robust standard errors by default.

### Correction Strategy
- Detected heteroskedasticity → use `cov_type='HC3'` in statsmodels
- Strong time-varying volatility → model it explicitly with GARCH
- Cross-sectional heteroskedasticity → Weighted Least Squares (WLS)

In [ ]:
# ── Inject heteroskedasticity into mock data for demonstration ────────────────
np.random.seed(7)
x_demo = np.linspace(1, 10, 300)
# Variance grows with x (heteroskedastic)
y_hetero = 2 + 0.8 * x_demo + np.random.normal(0, 0.3 * x_demo, 300)
# Constant variance (homoskedastic)
y_homo   = 2 + 0.8 * x_demo + np.random.normal(0, 1.5, 300)

X_demo = sm.add_constant(x_demo)

ols_hetero = sm.OLS(y_hetero, X_demo).fit()
ols_homo   = sm.OLS(y_homo,   X_demo).fit()

# Breusch-Pagan test
bp_lm_h, bp_pv_h, _, _ = het_breuschpagan(ols_hetero.resid, X_demo)
bp_lm_o, bp_pv_o, _, _ = het_breuschpagan(ols_homo.resid,   X_demo)

print("Breusch-Pagan Test:")
print(f"  Heteroskedastic model : LM={bp_lm_h:.3f}, p={bp_pv_h:.4f} {'⚠ HETERO' if bp_pv_h<0.05 else '✓'}")
print(f"  Homoskedastic model   : LM={bp_lm_o:.3f}, p={bp_pv_o:.4f} {'⚠ HETERO' if bp_pv_o<0.05 else '✓ HOMO'}")

# OLS vs Robust standard errors comparison
ols_robust = ols_hetero.get_robustcov_results(cov_type='HC3')
print("\nOLS vs Robust (HC3) Standard Errors on Heteroskedastic Data:")
print(f"{'':20} {'OLS SE':>12} {'HC3 SE':>12} {'Difference':>12}")
for i, name in enumerate(['const', 'x']):
    se_ols = ols_hetero.bse[i]
    se_hc3 = ols_robust.bse[i]
    print(f"{name:20} {se_ols:>12.5f} {se_hc3:>12.5f} {(se_hc3-se_ols)/se_ols*100:>11.1f}%")

# ── Visualisation ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, resid, title, color in [
    (axes[0], ols_hetero.resid, 'Heteroskedastic Residuals\n(fan shape = variance grows with x)', 'crimson'),
    (axes[1], ols_homo.resid,   'Homoskedastic Residuals\n(uniform spread = assumption holds)', 'steelblue')
]:
    ax.scatter(x_demo, resid, alpha=0.5, s=15, color=color)
    ax.axhline(0, color='black', linewidth=1.5, linestyle='--')
    ax.set_xlabel('x', fontweight='bold')
    ax.set_ylabel('Residual', fontweight='bold')
    ax.set_title(title, fontweight='bold')
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()


## 7. Cross-Sectional Regression — Fama-MacBeth Procedure

### What Is It?
**Fama-MacBeth (1973)** is the standard method for estimating risk premia from cross-sectional data while correctly accounting for cross-sectional correlation in residuals. Two-pass procedure:
1. **First pass**: Run time-series regression for each asset to estimate factor loadings $\hat{\beta}_i$
2. **Second pass**: Each period, regress cross-sectional returns on those loadings:  $R_{i,t} = \gamma_{0,t} + \gamma_{1,t}\hat{\beta}_i + u_{i,t}$
3. **Inference**: Average the $\gamma_{1,t}$ estimates and use their time-series standard errors

### Why Quants Need It
- **Factor Risk Premium Estimation**: The fundamental question "Is value risk actually rewarded?" is answered by Fama-MacBeth. A positive, significant $\gamma_{\text{HML}}$ means value stocks command a risk premium.
- **Quantitative Equity Signal Testing**: Before deploying a new alpha signal (e.g., short interest ratio, ESG score, sentiment), quant researchers run Fama-MacBeth regressions to check whether the signal predicts forward returns *after* controlling for known factors.
- **Asset Pricing Model Validation**: The CAPM predicts $\gamma_0=R_f$, $\gamma_1=E(R_m)-R_f$, and no other factors matter. Fama-MacBeth tests this directly — the finding that $\gamma_0 > R_f$ was the original evidence of CAPM's failure.
- **Regulatory Stress Testing**: The EBA and Fed stress test teams run cross-sectional regressions of bank equity returns on macro scenarios across multiple institutions per period — structurally identical to Fama-MacBeth.

In [ ]:
# ── Cross-sectional OLS: fundamental factors → forward returns ─────────────────
cs_features = ['log_pe', 'log_pb', 'roe', 'debt_equity', 'log_mktcap']
X_cs = sm.add_constant(df_cross[cs_features].values)
y_cs = df_cross['fw_return'].values

ols_cs = sm.OLS(y_cs, X_cs).fit()
print("=" * 65)
print("CROSS-SECTIONAL REGRESSION: FUNDAMENTAL FACTORS → FWD RETURN")
print("=" * 65)
print(ols_cs.summary())

# ── Fama-MacBeth simulation: repeat cross-section over T=50 periods ────────────
T = 50  # simulated months
fm_gammas = np.zeros((T, len(cs_features) + 1))

for t in range(T):
    # Slightly perturb forward returns to simulate cross-period variation
    y_t = y_cs + np.random.normal(0, 0.01, len(y_cs))
    ols_t = sm.OLS(y_t, X_cs).fit()
    fm_gammas[t] = ols_t.params

# Fama-MacBeth estimates: mean of period-by-period coefficients
fm_mean = fm_gammas.mean(axis=0)
fm_se   = fm_gammas.std(axis=0) / np.sqrt(T)   # Newey-West would improve this
fm_tstat = fm_mean / fm_se

param_names = ['const'] + cs_features
print("\n" + "=" * 60)
print("FAMA-MACBETH SECOND-PASS RISK PREMIA")
print("=" * 60)
print(f"{'Factor':<15} {'γ (mean)':>12} {'SE':>10} {'t-stat':>10} {'Sig':>6}")
print("-" * 60)
for name, g, se, t in zip(param_names, fm_mean, fm_se, fm_tstat):
    sig = '***' if abs(t) > 3 else '**' if abs(t) > 2.58 else '*' if abs(t) > 1.96 else ''
    print(f"{name:<15} {g:>12.6f} {se:>10.6f} {t:>10.3f} {sig:>6}")

# ── Visualisation ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Coefficient bar plot
ax = axes[0]
colors = ['green' if t > 1.96 else 'red' if t < -1.96 else 'gray' for t in fm_tstat[1:]]
ax.bar(cs_features, fm_mean[1:], color=colors, alpha=0.7)
ax.errorbar(cs_features, fm_mean[1:], yerr=1.96*fm_se[1:], fmt='none',
            color='black', capsize=5, linewidth=2)
ax.axhline(0, color='black', linestyle='--', linewidth=0.8)
ax.set_title('Fama-MacBeth Risk Premia γ\n(green=significant, gray=not)', fontweight='bold')
ax.set_ylabel('γ (Risk Premium)', fontweight='bold')
ax.tick_params(axis='x', rotation=20)
ax.grid(alpha=0.3, axis='y')

# Scatter: actual vs fitted
ax = axes[1]
ax.scatter(ols_cs.fittedvalues, y_cs, alpha=0.6, s=40, color='steelblue')
lim = max(abs(ols_cs.fittedvalues).max(), abs(y_cs).max())
ax.plot([-lim, lim], [-lim, lim], 'r--', linewidth=2)
ax.set_xlabel('Fitted Forward Return', fontweight='bold')
ax.set_ylabel('Actual Forward Return', fontweight='bold')
ax.set_title(f'Cross-Sectional Fit  (R²={ols_cs.rsquared:.3f})', fontweight='bold')
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()


## 8. Ridge Regression (L2 Regularization)

### What Is It?
Ridge adds a penalty on the size of coefficients to the OLS loss function, shrinking them toward zero but never exactly to zero:
$$\hat{\boldsymbol{\beta}}_{\text{ridge}} = \underset{\beta}{\arg\min} \left[ \|y - X\beta\|^2 + \lambda\|\beta\|^2 \right]$$

The closed-form solution is: $\hat{\beta}_{\text{ridge}} = (X^TX + \lambda I)^{-1}X^T y$

### Why Quants Need It
- **Highly Collinear Factor Models**: When your factor model has 20 highly correlated macro variables, OLS produces wildly unstable betas. Ridge shrinks and stabilises them, giving more reliable out-of-sample predictions. Major risk data providers use regularized regression for their macro factor overlay models.
- **Return Prediction with Many Features**: A machine learning quant model that predicts next-month stock returns from 200 financial ratios will massively overfit with OLS. Ridge provides a principled way to use all 200 variables without selecting among them.
- **Portfolio Construction with Short-Sale Constraints**: When building a minimum-variance portfolio with many assets, the covariance matrix is often near-singular. Ridge regression on the portfolio weights is mathematically equivalent to adding a diagonal shrinkage term to the covariance matrix — a standard technique in Ledoit-Wolf and Black-Litterman frameworks.
- **Bond Yield Curve Fitting**: The Nelson-Siegel model fits a smooth yield curve to observed bond yields. Ridge regression is used to fit the curve parameters, preventing overfitting when many bonds are available at nearby maturities.

**Ridge vs OLS Tradeoff:** As λ increases, bias increases but variance decreases. The optimal λ is chosen by cross-validation.

In [ ]:
# ── Ridge Regression on cross-sectional fundamental factors ───────────────────
scaler = StandardScaler()
X_std = scaler.fit_transform(df_cross[cs_features].values)
y_std = df_cross['fw_return'].values

# OLS coefficients (λ=0)
ols_lr = LinearRegression(fit_intercept=True)
ols_lr.fit(X_std, y_std)

# Ridge for a range of λ values
lambdas = np.logspace(-3, 3, 100)
ridge_coefs = np.zeros((len(lambdas), len(cs_features)))
ridge_cv_scores = []

for i, lam in enumerate(lambdas):
    ridge = Ridge(alpha=lam, fit_intercept=True)
    ridge.fit(X_std, y_std)
    ridge_coefs[i] = ridge.coef_
    cv = cross_val_score(ridge, X_std, y_std, cv=5, scoring='neg_mean_squared_error')
    ridge_cv_scores.append(-cv.mean())

best_lambda = lambdas[np.argmin(ridge_cv_scores)]
ridge_best = Ridge(alpha=best_lambda, fit_intercept=True)
ridge_best.fit(X_std, y_std)

print(f"Optimal λ (5-fold CV): {best_lambda:.4f}")
print(f"\n{'Feature':<15} {'OLS coef':>12} {'Ridge coef':>12} {'Shrinkage':>12}")
print("-" * 52)
for name, ols_c, ridge_c in zip(cs_features, ols_lr.coef_, ridge_best.coef_):
    shrink = (1 - abs(ridge_c)/abs(ols_c)) * 100 if abs(ols_c) > 1e-10 else 0
    print(f"{name:<15} {ols_c:>12.6f} {ridge_c:>12.6f} {shrink:>10.1f}%")

# ── Visualisation ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Ridge path (coefficient shrinkage as λ grows)
ax = axes[0]
for j, name in enumerate(cs_features):
    ax.plot(np.log10(lambdas), ridge_coefs[:, j], linewidth=2, label=name)
ax.axvline(np.log10(best_lambda), color='black', linestyle='--', linewidth=2,
           label=f'Optimal λ={best_lambda:.3f}')
ax.set_xlabel('log₁₀(λ)', fontweight='bold')
ax.set_ylabel('Coefficient', fontweight='bold')
ax.set_title('Ridge Regularization Path\n(coefficients shrink as λ increases)', fontweight='bold')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

# CV MSE vs λ
ax = axes[1]
ax.plot(np.log10(lambdas), ridge_cv_scores, color='steelblue', linewidth=2)
ax.axvline(np.log10(best_lambda), color='red', linestyle='--', linewidth=2,
           label=f'Optimal λ={best_lambda:.3f}')
ax.set_xlabel('log₁₀(λ)', fontweight='bold')
ax.set_ylabel('5-Fold CV MSE', fontweight='bold')
ax.set_title('Cross-Validation: Choosing Optimal λ', fontweight='bold')
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()


## 9. Lasso Regression (L1 Regularization) — Automatic Factor Selection

### What Is It?
Lasso (Least Absolute Shrinkage and Selection Operator) uses L1 penalty, which drives some coefficients exactly to zero — performing automatic feature selection:
$$\hat{\boldsymbol{\beta}}_{\text{lasso}} = \underset{\beta}{\arg\min}\left[\|y - X\beta\|^2 + \lambda\|\beta\|_1\right]$$

Unlike Ridge, Lasso produces **sparse** solutions: only a subset of predictors survive.

### Why Quants Need It
- **Automatic Signal Selection**: A quant researcher tests 200 candidate alpha signals. Running Lasso automatically identifies the ~15 that genuinely predict forward returns, discarding the other 185. This is far superior to manual t-stat screening, which suffers from multiple comparison bias.
- **Macro Factor Selection**: In building a macro risk model for sovereign bonds, Lasso selects from 40+ macro indicators (GDP growth, CPI, unemployment, PMI, credit spreads, etc.) the ~8 that actually drive yield changes. The sparse model is more interpretable and more robust out-of-sample.
- **High-Dimensional Credit Scoring**: Retail banks use Lasso to build credit scoring models from hundreds of bureau variables. Lasso identifies the sparse subset of variables with genuine predictive power, critical for regulatory interpretability (ECOA, Fair Lending).
- **Text/NLP Factor Models**: Quant funds (AQR, Two Sigma) build return prediction models from thousands of NLP features extracted from earnings calls and analyst reports. Lasso selects the sparse set of linguistic features that genuinely predict returns.

**Ridge vs Lasso — When to Use Which:**
| | Ridge | Lasso |
|--|-------|-------|
| Coefficient behavior | Shrinks toward 0, never zero | Sets some exactly to zero |
| Best when | Many small effects | Few large effects |
| Interpretability | All factors survive | Sparse: easy to interpret |
| Collinear features | Distributes weight | Picks one, drops others |

In [ ]:
from sklearn.linear_model import Lasso, LassoCV

# ── Lasso Regression — Feature Selection on cross-sectional data ──────────────
np.random.seed(42)

# Use cross-sectional data: 100 stocks, 5 features
# True model: only log_pe and log_pb matter significantly; others are noise
y_cs = df_cross['fw_return'].values
X_cs = df_cross[cs_features].values
X_cs_std = StandardScaler().fit_transform(X_cs)

# --- 1. Lasso regularization path ---
lambdas_lasso = np.logspace(-4, 1, 100)
lasso_coefs = []
for a in lambdas_lasso:
    lasso = Lasso(alpha=a, max_iter=5000)
    lasso.fit(X_cs_std, y_cs)
    lasso_coefs.append(lasso.coef_)
lasso_coefs = np.array(lasso_coefs)

# --- 2. LassoCV for optimal lambda ---
lasso_cv = LassoCV(alphas=lambdas_lasso, cv=5, max_iter=5000)
lasso_cv.fit(X_cs_std, y_cs)
best_lasso_lambda = lasso_cv.alpha_
final_lasso = Lasso(alpha=best_lasso_lambda, max_iter=5000)
final_lasso.fit(X_cs_std, y_cs)

# --- 3. OLS, Ridge, Lasso coefficient comparison ---
from sklearn.linear_model import LinearRegression, Ridge as SkRidge
ols_sk = LinearRegression().fit(X_cs_std, y_cs)
ridge_cv_sk = SkRidge(alpha=best_lambda if 'best_lambda' in dir() else 1.0)
ridge_cv_sk.fit(X_cs_std, y_cs)

coef_compare = pd.DataFrame({
    'Feature': cs_features,
    'OLS': ols_sk.coef_,
    'Ridge': ridge_cv_sk.coef_,
    'Lasso': final_lasso.coef_
})

# ── Plots ─────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Lasso Regression — Automatic Factor Selection', fontsize=15, fontweight='bold')

# Plot 1: Lasso regularization path
colors = plt.cm.tab10(np.linspace(0, 1, len(cs_features)))
for i, (feat, col) in enumerate(zip(cs_features, colors)):
    axes[0].plot(np.log10(lambdas_lasso), lasso_coefs[:, i], label=feat, color=col, lw=2)
axes[0].axvline(np.log10(best_lasso_lambda), color='red', linestyle='--', lw=1.5, label=f'CV λ={best_lasso_lambda:.4f}')
axes[0].set_xlabel('log₁₀(λ)', fontsize=11)
axes[0].set_ylabel('Coefficient', fontsize=11)
axes[0].set_title('Lasso Regularization Path\n(Coefficients → Exactly 0)', fontsize=11)
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3)

# Highlight: when each coefficient hits zero
for i, feat in enumerate(cs_features):
    zero_idx = np.where(lasso_coefs[:, i] == 0)[0]
    if len(zero_idx) > 0:
        axes[0].annotate('→0', xy=(np.log10(lambdas_lasso[zero_idx[0]]),
                                    lasso_coefs[zero_idx[0]-1, i]),
                          fontsize=7, color=colors[i])

# Plot 2: OLS vs Ridge vs Lasso coefficients
x_pos = np.arange(len(cs_features))
width = 0.25
bars1 = axes[1].bar(x_pos - width, coef_compare['OLS'], width, label='OLS', color='steelblue', alpha=0.85)
bars2 = axes[1].bar(x_pos, coef_compare['Ridge'], width, label='Ridge', color='darkorange', alpha=0.85)
bars3 = axes[1].bar(x_pos + width, coef_compare['Lasso'], width, label='Lasso', color='seagreen', alpha=0.85)
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels([f.replace('_', '\n') for f in cs_features], fontsize=9)
axes[1].set_ylabel('Standardized Coefficient', fontsize=11)
axes[1].set_title('OLS vs Ridge vs Lasso\nCoefficient Comparison', fontsize=11)
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis='y')

# Plot 3: Sparsity — number of nonzero coefficients vs lambda
n_nonzero = np.sum(lasso_coefs != 0, axis=1)
axes[2].plot(np.log10(lambdas_lasso), n_nonzero, color='purple', lw=2.5)
axes[2].axvline(np.log10(best_lasso_lambda), color='red', linestyle='--', lw=1.5,
                label=f'Optimal λ\n({int(np.sum(final_lasso.coef_ != 0))} features selected)')
axes[2].set_xlabel('log₁₀(λ)', fontsize=11)
axes[2].set_ylabel('Number of Non-Zero Coefficients', fontsize=11)
axes[2].set_title('Lasso Sparsity Path\n(Feature Elimination)', fontsize=11)
axes[2].set_yticks(range(0, len(cs_features) + 1))
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Summary table
print("\n=== OLS vs Ridge vs Lasso Coefficient Comparison ===")
print(f"{'Feature':<15} {'OLS':>10} {'Ridge':>10} {'Lasso':>10}")
print("─" * 50)
for _, row in coef_compare.iterrows():
    lasso_str = f"{row['Lasso']:10.4f}" if row['Lasso'] != 0 else "     ZERO "
    print(f"{row['Feature']:<15} {row['OLS']:10.4f} {row['Ridge']:10.4f} {lasso_str}")
print(f"\nLasso selected {int(np.sum(final_lasso.coef_ != 0))} out of {len(cs_features)} features at λ={best_lasso_lambda:.4f}")

## 10. Rolling Regression — Time-Varying Betas & Structural Breaks

### What Is It?
Rolling regression estimates model coefficients over a sliding window of length $W$, producing a *time series* of estimates:
$$\hat{\beta}_t = \left(X_{t-W:t}^\top X_{t-W:t}\right)^{-1} X_{t-W:t}^\top y_{t-W:t}$$

This captures how relationships change over time — static OLS gives a single average, rolling OLS reveals the evolution.

### Why Quants Need It
- **Beta Instability**: A stock's market beta is not constant. During the 2008 GFC, defensive stocks that had β≈0.3 in normal times suddenly behaved with β>1.0 as liquidity crises drove correlations toward 1. Static OLS misses this entirely. Rolling betas power risk systems like MSCI Barra's short-horizon factor models.
- **Style Drift Detection**: Hedge fund due diligence uses rolling Fama-French regressions to detect style drift — a fund marketed as "value/small-cap" gradually showing large growth loadings in its rolling betas is misrepresenting its strategy.
- **Regime Detection / Chow Test**: Rolling regression residuals spike sharply at structural breaks (the 2000 dot-com bust, 2008 GFC, 2020 COVID crash). Academic techniques like the Chow test formally test whether coefficients changed at a known breakpoint.
- **Options Delta Hedging**: Options desks compute time-varying hedge ratios (delta, delta-adjusted beta) continuously. Today's beta is what matters for today's hedge, not the 5-year average.

**Window Size Tradeoff:**
| Window | Responsiveness | Noise | Signal |
|--------|---------------|-------|--------|
| 20 days | Very fast | High | Unstable |
| 60 days | Moderate | Medium | Balanced |
| 252 days | Slow | Low | Stable but stale |

In [ ]:
from statsmodels.regression.rolling import RollingOLS

# ── Rolling Regression — Time-Varying Betas ───────────────────────────────────
# Simulate two regimes: calm period then crisis with beta shift
np.random.seed(42)
n_days = 500
dates = pd.date_range('2020-01-01', periods=n_days, freq='B')

# Market factor
mkt = np.random.normal(0, 0.01, n_days)

# Stock return with a structural break at day 300 (regime shift)
# Pre-break: alpha=0.0002, beta=0.8
# Post-break: alpha=-0.0005, beta=1.5  (crisis: beta spikes)
break_point = 300
true_alpha = np.where(np.arange(n_days) < break_point, 0.0002, -0.0005)
true_beta  = np.where(np.arange(n_days) < break_point, 0.8, 1.5)
stock_roll = true_alpha + true_beta * mkt + np.random.normal(0, 0.008, n_days)

df_roll = pd.DataFrame({'date': dates, 'stock': stock_roll, 'market': mkt}).set_index('date')

# --- Static OLS baseline ---
X_static = sm.add_constant(df_roll['market'])
static_model = sm.OLS(df_roll['stock'], X_static).fit()
static_alpha = static_model.params['const']
static_beta  = static_model.params['market']

# --- Rolling OLS with window=60 and window=120 ---
windows = [60, 120]
rolling_results = {}
for w in windows:
    roll_model = RollingOLS(df_roll['stock'], sm.add_constant(df_roll['market']), window=w)
    roll_res = roll_model.fit()
    rolling_results[w] = {
        'alpha': roll_res.params['const'],
        'beta':  roll_res.params['market']
    }

# ── Plots ─────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(14, 12))
fig.suptitle('Rolling Regression — Time-Varying Factor Exposures', fontsize=15, fontweight='bold')

# Plot 1: Return series with regime break annotation
axes[0].plot(df_roll.index, df_roll['stock'], color='navy', alpha=0.7, linewidth=0.8, label='Stock Return')
axes[0].plot(df_roll.index, df_roll['market'], color='orange', alpha=0.6, linewidth=0.8, label='Market Return')
axes[0].axvline(df_roll.index[break_point], color='red', linestyle='--', lw=2, label='Regime Break (Day 300)')
axes[0].set_title('Return Series with Structural Break', fontsize=11)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot 2: Rolling Beta (60-day and 120-day) vs true beta and static
axes[1].plot(df_roll.index, rolling_results[60]['beta'],
             color='dodgerblue', lw=1.5, label='Rolling β (60-day)')
axes[1].plot(df_roll.index, rolling_results[120]['beta'],
             color='purple', lw=1.5, linestyle='--', label='Rolling β (120-day)')
axes[1].axhline(static_beta, color='red', linestyle=':', lw=2, label=f'Static OLS β = {static_beta:.2f}')
axes[1].axhline(0.8, color='seagreen', linestyle='--', lw=1, alpha=0.7, label='True β pre-break = 0.8')
axes[1].axhline(1.5, color='tomato', linestyle='--', lw=1, alpha=0.7, label='True β post-break = 1.5')
axes[1].axvline(df_roll.index[break_point], color='red', linestyle='--', lw=2, alpha=0.5)
axes[1].set_title('Rolling Market Beta: Detecting the Regime Shift', fontsize=11)
axes[1].set_ylabel('Beta (Market Sensitivity)', fontsize=10)
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3)

# Plot 3: Rolling Alpha
axes[2].plot(df_roll.index, rolling_results[60]['alpha'],
             color='darkorange', lw=1.5, label='Rolling α (60-day)')
axes[2].plot(df_roll.index, rolling_results[120]['alpha'],
             color='teal', lw=1.5, linestyle='--', label='Rolling α (120-day)')
axes[2].axhline(static_alpha, color='red', linestyle=':', lw=2, label=f'Static OLS α = {static_alpha:.5f}')
axes[2].axhline(0.0002, color='seagreen', linestyle='--', lw=1, alpha=0.7, label='True α pre-break')
axes[2].axhline(-0.0005, color='tomato', linestyle='--', lw=1, alpha=0.7, label='True α post-break')
axes[2].axvline(df_roll.index[break_point], color='red', linestyle='--', lw=2, alpha=0.5)
axes[2].set_title('Rolling Alpha: Jensen\'s Alpha Over Time', fontsize=11)
axes[2].set_ylabel('Alpha (Intercept)', fontsize=10)
axes[2].legend(fontsize=8)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Summary statistics
print("=== Static OLS vs Rolling Regression Summary ===")
print(f"\nStatic OLS:  Alpha = {static_alpha:.5f}  |  Beta = {static_beta:.4f}")
print(f"\nTrue Parameters:")
print(f"  Pre-break  (days 0–299):  alpha=0.0002, beta=0.80")
print(f"  Post-break (days 300+):   alpha=-0.0005, beta=1.50")
print(f"\nRolling (60d) at break point (day 300):")
idx300 = 310  # a bit after break
print(f"  Alpha = {rolling_results[60]['alpha'].iloc[idx300]:.5f}  |  Beta = {rolling_results[60]['beta'].iloc[idx300]:.4f}")
print(f"\nConclusion: Rolling regression captures the beta spike from 0.8 → 1.5.")
print("Static OLS 'averages away' the regime shift, masking the true risk.")

## 11. Principal Component Regression (PCR) — Dimensionality Reduction

### What Is It?
Principal Component Regression combines PCA with OLS. Instead of regressing on correlated raw predictors, we:
1. Decompose $X$ into orthogonal principal components $Z = X V_k$ (first $k$ PCs)
2. Regress $y$ on $Z$: solve $\hat{\gamma} = (Z^\top Z)^{-1} Z^\top y$
3. Back-transform: $\hat{\beta}_{\text{PCR}} = V_k \hat{\gamma}$

This eliminates multicollinearity by construction — principal components are orthogonal by definition.

### Why Quants Need It
- **Macro Risk Models**: 30 correlated macro indicators (GDP, employment, PMI, industrial production, etc.) are all highly correlated. PCA extracts 3-5 orthogonal "macro factors" (typically: level, slope, business cycle, inflation) that explain 90% of variance. PCR builds sparse, interpretable models.
- **Yield Curve Modeling**: The Nelson-Siegel model is essentially PCR applied to bond yields. PCA of the yield curve reliably extracts 3 components that explain 99%+ of yield curve variance: **level** (PC1 — all maturities move), **slope** (PC2 — short vs long rates differ), and **curvature** (PC3 — belly vs wings).
- **Statistical Equity Factor Models**: Barra, Axioma, and Bloomberg equity risk models use PCA to extract statistical risk factors from large return covariance matrices (thousands of stocks). PCR on these statistical factors avoids the overfitting that plagues raw-feature regression.
- **High-Frequency Trading**: HFT firms run PCR on hundreds of microstructure signals (order flow imbalance, bid-ask spread, depth, etc.) to build orthogonal signal composites, reducing noise and improving Sharpe ratios.

**PCR vs Ridge vs Lasso:**
| | PCR | Ridge | Lasso |
|--|-----|-------|-------|
| Handles multicollinearity? | ✅ Exactly | ✅ Shrinks | ⚠️ Picks one |
| Sparse (feature selection)? | ❌ | ❌ | ✅ |
| Interpretability | Medium (PCs) | Medium | High (sparse) |
| Key parameter | # of components k | λ (shrinkage) | λ (sparsity) |

In [ ]:
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import cross_val_score

# ── Principal Component Regression (PCR) ──────────────────────────────────────
# Use cross-sectional data (100 stocks, 5 correlated fundamental factors)
y_pcr = df_cross['fw_return'].values
X_pcr = df_cross[cs_features].values

# Add more correlated features to make PCR relevant (5 → 10 features with structure)
np.random.seed(42)
n_stocks = len(df_cross)
X_extended = np.hstack([
    X_pcr,
    X_pcr[:, :3] + np.random.normal(0, 0.05, (n_stocks, 3)),   # near-duplicate features
    np.random.normal(0, 1, (n_stocks, 2))                        # pure noise features
])
feature_names_ext = cs_features + ['log_pe_v2', 'log_pb_v2', 'roe_v2', 'noise_1', 'noise_2']

scaler_pcr = StandardScaler()
X_pcr_std = scaler_pcr.fit_transform(X_extended)

# --- PCA: explained variance ---
pca_full = PCA()
pca_full.fit(X_pcr_std)
explained_var = pca_full.explained_variance_ratio_
cum_explained = np.cumsum(explained_var)

# --- PCR with different numbers of components ---
n_components_range = range(1, X_extended.shape[1] + 1)
pcr_r2 = []
pcr_mse = []
for k in n_components_range:
    pcr_pipe = Pipeline([('pca', PCA(n_components=k)), ('ols', LinearRegression())])
    cv_scores = cross_val_score(pcr_pipe, X_pcr_std, y_pcr, cv=5, scoring='r2')
    pcr_r2.append(cv_scores.mean())
    # MSE on full data
    pcr_pipe.fit(X_pcr_std, y_pcr)
    y_pred_pcr = pcr_pipe.predict(X_pcr_std)
    pcr_mse.append(mean_squared_error(y_pcr, y_pred_pcr))

best_k = int(np.argmax(pcr_r2)) + 1

# --- Fit best PCR ---
best_pcr = Pipeline([('pca', PCA(n_components=best_k)), ('ols', LinearRegression())])
best_pcr.fit(X_pcr_std, y_pcr)
y_pred_best = best_pcr.predict(X_pcr_std)

pca_best = best_pcr.named_steps['pca']
loadings = pd.DataFrame(pca_best.components_.T,
                         index=feature_names_ext,
                         columns=[f'PC{i+1}' for i in range(best_k)])

# --- OLS baseline on all features ---
ols_full = LinearRegression().fit(X_pcr_std, y_pcr)
ols_r2 = r2_score(y_pcr, ols_full.predict(X_pcr_std))

# ── Plots ─────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(15, 11))
fig.suptitle('Principal Component Regression (PCR)', fontsize=15, fontweight='bold')

# Plot 1: Scree plot — explained variance
ax = axes[0, 0]
ax.bar(range(1, len(explained_var)+1), explained_var * 100,
       color='steelblue', alpha=0.8, label='Individual')
ax2 = ax.twinx()
ax2.plot(range(1, len(cum_explained)+1), cum_explained * 100,
         color='tomato', marker='o', linewidth=2, label='Cumulative')
ax2.axhline(90, color='gray', linestyle='--', lw=1, alpha=0.7)
ax2.set_ylabel('Cumulative Explained Variance (%)', color='tomato', fontsize=10)
ax2.tick_params(axis='y', labelcolor='tomato')
ax.set_xlabel('Principal Component', fontsize=11)
ax.set_ylabel('Explained Variance (%)', fontsize=11)
ax.set_title('Scree Plot — PCA Explained Variance', fontsize=11)
# Combined legend
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, fontsize=9)
ax.grid(True, alpha=0.3)

# Plot 2: Cross-validated R² vs number of PCs
axes[0, 1].plot(list(n_components_range), pcr_r2, color='seagreen', marker='o', lw=2)
axes[0, 1].axvline(best_k, color='red', linestyle='--', lw=2, label=f'Optimal k={best_k}')
axes[0, 1].axhline(ols_r2, color='gray', linestyle=':', lw=2, label=f'Full OLS R²={ols_r2:.3f}')
axes[0, 1].set_xlabel('Number of Principal Components (k)', fontsize=11)
axes[0, 1].set_ylabel('Cross-Validated R²', fontsize=11)
axes[0, 1].set_title('PCR: Optimal Number of Components', fontsize=11)
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Plot 3: PC loadings heatmap (first 5 components)
n_show = min(5, best_k)
loadings_show = loadings.iloc[:, :n_show]
im = axes[1, 0].imshow(loadings_show.values.T, aspect='auto', cmap='RdBu_r',
                        vmin=-1, vmax=1)
plt.colorbar(im, ax=axes[1, 0])
axes[1, 0].set_xticks(range(len(feature_names_ext)))
axes[1, 0].set_xticklabels([f.replace('_', '\n') for f in feature_names_ext], fontsize=7)
axes[1, 0].set_yticks(range(n_show))
axes[1, 0].set_yticklabels([f'PC{i+1}' for i in range(n_show)], fontsize=9)
axes[1, 0].set_title(f'PC Loadings Heatmap (first {n_show} components)', fontsize=11)
for i in range(n_show):
    for j in range(len(feature_names_ext)):
        axes[1, 0].text(j, i, f'{loadings_show.values[j, i]:.2f}',
                        ha='center', va='center', fontsize=6,
                        color='white' if abs(loadings_show.values[j, i]) > 0.5 else 'black')

# Plot 4: Actual vs Fitted (PCR vs OLS)
y_ols_pred = ols_full.predict(X_pcr_std)
axes[1, 1].scatter(y_pcr, y_pred_best, alpha=0.5, s=20, color='seagreen', label=f'PCR (k={best_k})')
axes[1, 1].scatter(y_pcr, y_ols_pred, alpha=0.3, s=20, color='steelblue', label='OLS (all features)')
lims = [min(y_pcr.min(), y_pred_best.min()), max(y_pcr.max(), y_pred_best.max())]
axes[1, 1].plot(lims, lims, 'r--', lw=2, label='Perfect fit')
axes[1, 1].set_xlabel('Actual Return', fontsize=11)
axes[1, 1].set_ylabel('Predicted Return', fontsize=11)
axes[1, 1].set_title('Actual vs Fitted Returns\n(PCR vs Full OLS)', fontsize=11)
axes[1, 1].legend(fontsize=9)
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Summary
print(f"=== PCR Results ===")
print(f"Total features: {X_extended.shape[1]}")
print(f"Optimal components (by CV R²): k = {best_k}")
print(f"Variance explained by {best_k} PCs: {cum_explained[best_k-1]*100:.1f}%")
print(f"\nIn-sample R²:  OLS (all {X_extended.shape[1]} features) = {ols_r2:.4f}")
print(f"               PCR (k={best_k} components)             = {r2_score(y_pcr, y_pred_best):.4f}")
print(f"\nKey insight: PCR achieves similar predictive accuracy with {best_k} orthogonal")
print(f"components instead of {X_extended.shape[1]} correlated features, reducing overfitting risk.")

## 12. Summary & Key Takeaways

### The Regression Toolkit for Quant Finance

This notebook covered 11 regression techniques. The table below summarizes when and why a practitioner would choose each approach:

| # | Method | Core Problem Solved | Best Quant Use Case | Key Assumption / Caveat |
|---|--------|--------------------|--------------------|------------------------|
| 1 | **Simple OLS / CAPM** | Linear factor exposure | Beta estimation, performance attribution | Assumes linear relationship, constant beta |
| 2 | **Multiple Linear Regression** | Multi-factor exposure | Fama-French 3/4/5-factor models | Correlated factors inflate standard errors |
| 3 | **OLS Diagnostics (Gauss-Markov)** | Validate model assumptions | Pre-deployment model validation | All 5 BLUE assumptions must hold for valid inference |
| 4 | **VIF / Multicollinearity** | Detect redundant factors | Factor model construction | VIF > 10 → unstable, unreliable coefficient estimates |
| 5 | **Heteroskedasticity** | Handle non-constant variance | High-vol regimes, options pricing | Use HC3 robust SEs; Breusch-Pagan test |
| 6 | **Fama-MacBeth** | Panel data with cross-sectional returns | Risk premia estimation | T accounts for cross-sectional/time dimensions |
| 7 | **Ridge Regression** | Multicollinearity + many small effects | Factor shrinkage in risk models | L2 penalty; never zeroes coefficients |
| 8 | **Lasso Regression** | Automatic feature selection | Signal selection from large alpha libraries | L1 penalty; sparse solutions |
| 9 | **Rolling Regression** | Time-varying parameters | Beta instability, style drift, regime changes | Window choice is critical tradeoff |
| 10 | **PCR** | Dimensionality reduction + orthogonality | Statistical risk models, yield curve | Components orthogonal but less interpretable |

---

### Decision Flowchart: Which Regression Should I Use?

```
START
  └─ Time series or cross-section?
       ├─ TIME SERIES
       │    └─ Beta stable over time?
       │         ├─ YES → OLS / Multiple Regression
       │         └─ NO  → Rolling Regression
       │
       └─ CROSS-SECTIONAL
            └─ Many predictors or multicollinearity?
                 ├─ NO (few clean factors) → OLS / Fama-MacBeth
                 ├─ YES + want all factors → Ridge Regression
                 ├─ YES + want sparse model → Lasso Regression
                 └─ YES + correlated factor groups → PCR
```

---

### Core Formulas Reference

| Concept | Formula |
|---------|---------|
| OLS | $\hat{\beta} = (X^\top X)^{-1} X^\top y$ |
| CAPM | $R_i - R_f = \alpha + \beta(R_m - R_f) + \varepsilon$ |
| VIF | $\text{VIF}_j = \frac{1}{1-R_j^2}$ |
| Ridge | $\hat{\beta}_R = (X^\top X + \lambda I)^{-1} X^\top y$ |
| Lasso | $\hat{\beta}_L = \arg\min\left[\|y-X\beta\|^2 + \lambda\|\beta\|_1\right]$ |
| Rolling OLS | $\hat{\beta}_t = (X_{t-W:t}^\top X_{t-W:t})^{-1} X_{t-W:t}^\top y_{t-W:t}$ |
| Fama-MacBeth | $\hat{\lambda}_k = \frac{1}{T}\sum_{t=1}^T \hat{\gamma}_{k,t}$ |

---

### Key Practical Insights

1. **Always check OLS assumptions** before trusting standard errors or p-values — heteroskedasticity and autocorrelation are extremely common in financial return data.
2. **Fama-MacBeth is the industry standard** for risk premia estimation in academic and institutional research.  
3. **Rolling regression reveals what static OLS hides** — regime changes, beta drift, and structural breaks are invisible to full-sample OLS.
4. **Ridge for shrinkage, Lasso for selection, PCR for orthogonality** — these are not competing methods but tools for different problems.
5. **High VIF is not always a problem**: if your goal is prediction only (not inference), multicollinearity does not bias forecasts — it only inflates standard errors, making coefficient interpretation unreliable.